# Comparação entre planilha Numbers e PDFs

Este notebook compara a planilha Numbers do semestre com os PDFs de oferta de turmas e gera o relatório `comparacao.md`.

**Arquivos lidos**
- Planilha Numbers localizada a partir do caminho lógico `Dados/`.
- `Dados/DSC.pdf`.
- `Dados/BCC_mat.pdf` (`BCC-M`).
- `Dados/BCC_not.pdf` (`BCC-N`).
- `Dados/BCD_not.pdf` (`BCD-N`).
- `Dados/SIS_not.pdf` (`SIS-N`).

**Validações executadas**
- A) Turmas dos cursos ausentes na planilha.
- B) Turmas do DSC ausentes na planilha.
- C) Verificações internas em todas as fontes:
  - total de créditos ímpar;
  - turmas em concentrado;
  - turmas sem professor.
- D) Conflitos de professor na planilha.
- E) Conflitos de espaço físico na planilha.
- F) Conflitos de semestre/fase na planilha.
- G) Diferenças entre planilha e PDFs por código.

**Relatório gerado**
- Arquivo Markdown `comparacao.md` no diretório de dados resolvido a partir de `Dados`.
- As comparações também são exibidas no próprio notebook com tabelas pandas e Markdown.

**Observações de implementação**
- O código resolve `Dados` tanto como diretório, link simbólico quanto alias do macOS.
- Os nomes das colunas da planilha são normalizados automaticamente.
- Quando uma fonte traz apenas crédito total, o notebook usa esse total como `Créditos Teóricos` e `0` como `Créditos Práticos` para manter a validação consistente.

In [1]:
from __future__ import annotations

import platform
import re
import subprocess
import unicodedata
from pathlib import Path

import pandas as pd
import pdfplumber
from IPython.display import Markdown, display
from numbers_parser import Document

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)

DAY_NAMES = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sab']
COURSE_FILE_MAP = {
    'BCC_mat.pdf': 'BCC-M',
    'BCC_not.pdf': 'BCC-N',
    'BCD_not.pdf': 'BCD-N',
    'SIS_not.pdf': 'SIS-N',
}
PLACEHOLDER_VALUES = {
    '_não_', '_nao_', '_psps', 'psps', 'não', 'nao', 'n/a', 'none', 'null', 'nan', '_não', '_nao'
}

print('Python:', platform.python_version())
print('Pandas:', pd.__version__)
print('Diretório atual:', Path.cwd())

Python: 3.14.0
Pandas: 3.0.3
Diretório atual: /Users/daltonreis/GitHub/DSC/dsc/___Python


In [2]:
def clean_text(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    text = str(value).replace(' ', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text or None


def ascii_upper(value):
    text = clean_text(value)
    if not text:
        return None
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    return text.upper()


def slug_text(value):
    text = ascii_upper(value)
    if not text:
        return None
    text = re.sub(r'[^A-Z0-9/]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip() or None


def normalize_code(value):
    text = clean_text(value)
    if not text:
        return None
    return re.sub(r'\s+', '', text).upper()


def normalize_professor(value):
    text = slug_text(value)
    if not text:
        return None
    compact = text.replace(' ', '').lower()
    if text.lower() in PLACEHOLDER_VALUES or compact in PLACEHOLDER_VALUES:
        return None
    return text


def course_tokens(text):
    cleaned = clean_text(text)
    if not cleaned:
        return set()
    return {part for part in [ascii_upper(piece).strip() for piece in re.split(r'[,;/]', cleaned)] if part}


def canonicalize_schedule_parts(parts):
    tokens = []
    for raw in parts:
        cleaned = clean_text(raw)
        if not cleaned:
            continue
        for item in cleaned.splitlines():
            token = clean_text(item)
            if not token:
                continue
            if token == 'C' and tokens:
                tokens[-1] = f'{tokens[-1]}C'
            else:
                tokens.append(token)
    ordered = []
    seen = set()
    for token in tokens:
        if token not in seen:
            ordered.append(token)
            seen.add(token)
    return ordered


def format_schedule(day_map):
    items = []
    for day in DAY_NAMES:
        tokens = canonicalize_schedule_parts(day_map.get(day, []))
        if tokens:
            items.append(f"{day} {' / '.join(tokens)}")
    return '; '.join(items) if items else None


def normalize_schedule(value):
    text = clean_text(value)
    return ascii_upper(text) if text else None


def is_concentrated(horario, flag=False):
    text = clean_text(horario) or ''
    tokens = re.findall(r'\d{1,2}/\d{1,2}C?', text.upper())
    return bool(flag) or text.upper().startswith('CONCENTRADO') or any(token.endswith('C') for token in tokens)


def normalized_header(value):
    text = ascii_upper(value) or ''
    return re.sub(r'[^A-Z0-9]+', '', text)


def pick_column(columns, aliases, required=True):
    header_map = {normalized_header(column): column for column in columns}
    for alias in aliases:
        key = normalized_header(alias)
        if key in header_map:
            return header_map[key]
    if required:
        raise KeyError(f'Nenhuma coluna compatível com {aliases} foi encontrada.')
    return None


def resolve_dados_path(path_str='Dados'):
    path = Path(path_str)
    if path.is_dir() or path.is_symlink():
        return path.resolve()
    if path.exists() and path.is_file() and platform.system() == 'Darwin':
        script = 'tell application "Finder" to POSIX path of ((original item of alias file ((POSIX file "%s") as text)) as alias)'
        command = script % str(path.resolve())
        result = subprocess.run(['osascript', '-e', command], check=True, capture_output=True, text=True)
        return Path(result.stdout.strip()).resolve()
    raise FileNotFoundError(f'Não foi possível resolver o caminho de entrada: {path_str}')


def find_numbers_file(data_root: Path) -> Path:
    candidates = []
    for root in [data_root, data_root.parent]:
        if root.exists():
            candidates.extend(root.glob('*.numbers'))
    candidates = sorted({candidate.resolve() for candidate in candidates}, key=lambda item: ('turmas' not in item.name.lower(), item.name.lower()))
    if not candidates:
        raise FileNotFoundError(f'Nenhum arquivo .numbers encontrado em {data_root} ou {data_root.parent}.')
    return candidates[0]


def load_numbers_spreadsheet(path: Path) -> pd.DataFrame:
    document = Document(path)
    if not document.sheets or not document.sheets[0].tables:
        raise ValueError(f'A planilha {path.name} não possui tabelas legíveis.')

    table = document.sheets[0].tables[0]
    rows = list(table.rows(values_only=True))
    if not rows:
        raise ValueError(f'A planilha {path.name} está vazia.')

    columns = [clean_text(column) or f'col_{idx}' for idx, column in enumerate(rows[0])]
    frame = pd.DataFrame(rows[1:], columns=columns)

    col_nome = pick_column(frame.columns, ['Nome', 'Turma'])
    col_codigo = pick_column(frame.columns, ['Código', 'Codigo'])
    col_curso = pick_column(frame.columns, ['Curso'])
    col_professor = pick_column(frame.columns, ['Professor', 'Docente'])
    col_espaco = pick_column(frame.columns, ['Espaço', 'Espaco', 'Sala'], required=False)
    col_fase = pick_column(frame.columns, ['Fase', 'Semestre'], required=False)
    col_grupo = pick_column(frame.columns, ['Grupo'], required=False)
    col_creditos_teoricos = pick_column(frame.columns, ['Créditos Teóricos', 'Creditos Teoricos'], required=False)
    col_creditos_praticos = pick_column(frame.columns, ['Créditos Práticos', 'Creditos Praticos'], required=False)
    col_creditos_total = pick_column(frame.columns, ['Cre.', 'Créditos', 'Creditos', 'Carga Horária'], required=False)
    col_concentrado = pick_column(frame.columns, ['Conc.', 'Concentrado'], required=False)
    day_columns = {day: pick_column(frame.columns, [day], required=False) for day in DAY_NAMES}

    result = pd.DataFrame({
        'Fonte': path.name,
        'source_kind': 'planilha',
        'Código': frame[col_codigo].map(normalize_code),
        'Nome': frame[col_nome].map(clean_text),
        'Curso': frame[col_curso].map(clean_text),
        'Professor': frame[col_professor].map(clean_text),
        'Espaço': frame[col_espaco].map(clean_text) if col_espaco else None,
        'Fase': pd.to_numeric(frame[col_fase], errors='coerce').astype('Int64') if col_fase else pd.Series([pd.NA] * len(frame), dtype='Int64'),
        'Grupo': frame[col_grupo].map(clean_text) if col_grupo else None,
    })

    result['Horário'] = frame.apply(
        lambda row: format_schedule({day: [row[column]] for day, column in day_columns.items() if column}),
        axis=1,
    )

    if col_creditos_teoricos or col_creditos_praticos:
        result['Créditos Teóricos'] = pd.to_numeric(frame[col_creditos_teoricos], errors='coerce').astype('Int64') if col_creditos_teoricos else 0
        result['Créditos Práticos'] = pd.to_numeric(frame[col_creditos_praticos], errors='coerce').astype('Int64') if col_creditos_praticos else 0
    elif col_creditos_total:
        total = pd.to_numeric(frame[col_creditos_total], errors='coerce').astype('Int64')
        result['Créditos Teóricos'] = total
        result['Créditos Práticos'] = 0
    else:
        raise KeyError('A planilha não possui colunas de créditos reconhecíveis.')

    result['Total de Créditos'] = result['Créditos Teóricos'].fillna(0) + result['Créditos Práticos'].fillna(0)
    result['concentrado_flag'] = frame[col_concentrado].fillna(False).astype(bool) if col_concentrado else False
    result.loc[result['Horário'].isna() & result['concentrado_flag'], 'Horário'] = 'Concentrado'
    return result


def merge_description_lines(text):
    descriptions = []
    for raw_line in str(text).splitlines():
        line = clean_text(raw_line)
        if not line:
            continue
        if re.match(r'^\d[A-Z]{3}\.\d{4}\.\d{2}\.\d{3}-\d\b', line):
            descriptions.append(line)
        elif descriptions:
            descriptions[-1] = f"{descriptions[-1]} {line}"
    return descriptions


def parse_course_description(description, course_code, source_name, schedule):
    match = re.match(r'^(?P<fase>\d)(?P<codigo>[A-Z]{3}\.\d{4}\.\d{2}\.\d{3}-\d)\s+(?P<body>.+)$', description)
    if not match:
        raise ValueError(f'Descrição do PDF não reconhecida: {description}')

    body = match.group('body')
    pair_matches = list(re.finditer(r'(?<!\S)(\d+)\s+(\d+)(?!\S)', body))
    single_matches = list(re.finditer(r'(?<!\S)(\d+)(?!\S)', body))

    if pair_matches:
        last = pair_matches[-1]
        total = int(last.group(2))
        body = clean_text(f"{body[:last.start()]} {body[last.end():]}")
    elif single_matches:
        last = single_matches[-1]
        total = int(last.group(1))
        body = clean_text(f"{body[:last.start()]} {body[last.end():]}")
    else:
        raise ValueError(f'Créditos não encontrados na descrição: {description}')

    parenthetical = re.match(r'(?P<nome>.+?)\s+\((?P<prof>[^()]*)\)$', body)
    nome = clean_text(parenthetical.group('nome')) if parenthetical else clean_text(body)
    professor = clean_text(parenthetical.group('prof')) if parenthetical else None

    return {
        'Fonte': source_name,
        'source_kind': 'pdf_curso',
        'Código': normalize_code(match.group('codigo')),
        'Nome': nome,
        'Curso': course_code,
        'Professor': professor,
        'Horário': schedule,
        'Espaço': None,
        'Fase': int(match.group('fase')),
        'Grupo': None,
        'Créditos Teóricos': total,
        'Créditos Práticos': 0,
        'Total de Créditos': total,
        'concentrado_flag': is_concentrated(schedule),
    }


def load_course_pdf(path: Path, course_code: str) -> pd.DataFrame:
    records = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            for table in page.extract_tables():
                if not table or len(table[0]) != 7:
                    continue
                rows = table[2:]
                index = 0
                while index < len(rows):
                    first_cell = rows[index][0]
                    if not first_cell or not re.search(r'\d[A-Z]{3}\.\d{4}\.\d{2}\.\d{3}-\d', first_cell):
                        index += 1
                        continue
                    descriptions = merge_description_lines(first_cell)
                    block_rows = rows[index:index + len(descriptions)]
                    if len(block_rows) < len(descriptions):
                        block_rows += [[None, '', '', '', '', '', '']] * (len(descriptions) - len(block_rows))
                    for offset, description in enumerate(descriptions):
                        line = block_rows[offset]
                        schedule = format_schedule({DAY_NAMES[column - 1]: [line[column]] for column in range(1, 7)})
                        records.append(parse_course_description(description, course_code, path.name, schedule))
                    index += len(descriptions)
    return pd.DataFrame(records)


def parse_dsc_text_blocks(page_text: str):
    blocks = []
    current = None
    ignored_prefixes = (
        'DIVISÃO DE REGISTROS',
        'Registros Acadêmicos',
        'Turma por Depto',
        'Departamento:',
        'Alu FaGru',
        'Créditos Horário',
        'Turma Código Professor',
        'Núcleo de Informática',
    )
    for raw_line in page_text.splitlines():
        line = clean_text(raw_line)
        if not line or line.startswith(ignored_prefixes):
            continue
        if re.match(r'^[A-Z]{3}\.\s*\d{4}\.\s*\d{2}\.\s*\d{3}-\d\b', line):
            current = {'first_line': line, 'extra_courses': []}
            blocks.append(current)
        elif current and re.match(r'^[A-Z]{2,4}-[A-Z]\s+\d+\s+[A-Z0-9]+$', line):
            current['extra_courses'].append(line)
    return blocks


def parse_dsc_block(block, table_row, source_name):
    match = re.match(r'^(?P<codigo>[A-Z]{3}\.\s*\d{4}\.\s*\d{2}\.\s*\d{3}-\d)\s+(?P<rest>.+)$', block['first_line'])
    if not match:
        raise ValueError(f'Linha do PDF DSC não reconhecida: {block["first_line"]}')

    rest = match.group('rest')
    professor = clean_text(table_row['Professor'])
    creditos_teoricos = pd.to_numeric(table_row['Créditos Teóricos'], errors='coerce')
    creditos_praticos = pd.to_numeric(table_row['Créditos Práticos'], errors='coerce')

    triplet_regex = re.compile(r'(?P<curso>[A-Z]{2,4}-[A-Z])\s+(?P<fase>\d+)\s+(?P<grupo>[A-Z0-9]+)$')
    triplets = []
    suffix = triplet_regex.search(rest)
    prefix = rest
    if suffix:
        triplets.append((suffix.group('curso'), int(suffix.group('fase')), suffix.group('grupo')))
        prefix = rest[:suffix.start()].rstrip()
    for extra in block['extra_courses']:
        extra_match = triplet_regex.match(extra)
        if extra_match:
            triplets.append((extra_match.group('curso'), int(extra_match.group('fase')), extra_match.group('grupo')))

    if pd.notna(creditos_teoricos) and pd.notna(creditos_praticos):
        prefix = re.sub(rf'\s+{int(creditos_teoricos)}\s+{int(creditos_praticos)}\b.*$', '', prefix).rstrip()

    if professor:
        prefix = re.sub(rf'\s*\d*\s*{re.escape(professor)}\s*$', '', prefix).rstrip()

    horario = format_schedule({day: [table_row[day]] for day in DAY_NAMES})
    return {
        'Fonte': source_name,
        'source_kind': 'pdf_dsc',
        'Código': normalize_code(match.group('codigo')),
        'Nome': clean_text(prefix),
        'Curso': ', '.join(dict.fromkeys(course for course, _, _ in triplets)) if triplets else None,
        'Professor': professor,
        'Horário': horario,
        'Espaço': None,
        'Fase': triplets[0][1] if triplets else pd.NA,
        'Grupo': triplets[0][2] if triplets else None,
        'Créditos Teóricos': int(creditos_teoricos) if pd.notna(creditos_teoricos) else pd.NA,
        'Créditos Práticos': int(creditos_praticos) if pd.notna(creditos_praticos) else pd.NA,
        'Total de Créditos': (int(creditos_teoricos) if pd.notna(creditos_teoricos) else 0) + (int(creditos_praticos) if pd.notna(creditos_praticos) else 0),
        'concentrado_flag': is_concentrated(horario),
    }


def load_dsc_pdf(path: Path) -> pd.DataFrame:
    records = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables()
            if not tables:
                continue
            table_rows = [
                dict(zip(['Professor', 'Créditos Teóricos', 'Créditos Práticos', 'Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sab'], row))
                for row in tables[0]
            ]
            blocks = parse_dsc_text_blocks(page.extract_text() or '')
            if len(blocks) != len(table_rows):
                raise ValueError(
                    f'Página {page.page_number} do DSC.pdf com contagem divergente: {len(blocks)} blocos x {len(table_rows)} linhas de tabela.'
                )
            records.extend(parse_dsc_block(block, row, path.name) for block, row in zip(blocks, table_rows))
    return pd.DataFrame(records)


def prepare_dataset(frame: pd.DataFrame) -> pd.DataFrame:
    prepared = frame.copy()
    prepared['Código'] = prepared['Código'].map(normalize_code)
    for column in ['Nome', 'Curso', 'Professor', 'Horário', 'Espaço', 'Grupo']:
        prepared[column] = prepared[column].map(clean_text)
    prepared['nome_norm'] = prepared['Nome'].map(slug_text)
    prepared['professor_norm'] = prepared['Professor'].map(normalize_professor)
    prepared['horario_norm'] = prepared['Horário'].map(normalize_schedule)
    prepared['curso_tokens'] = prepared['Curso'].map(course_tokens)
    prepared['composite_key'] = prepared.apply(
        lambda row: f"{row['Código']}|{row['nome_norm']}" if row['Código'] and row['nome_norm'] else None,
        axis=1,
    )
    prepared['concentrado_flag'] = prepared.apply(lambda row: is_concentrated(row['Horário'], row['concentrado_flag']), axis=1)
    prepared['Créditos Teóricos'] = pd.to_numeric(prepared['Créditos Teóricos'], errors='coerce').astype('Int64')
    prepared['Créditos Práticos'] = pd.to_numeric(prepared['Créditos Práticos'], errors='coerce').astype('Int64')
    prepared['Total de Créditos'] = pd.to_numeric(prepared['Total de Créditos'], errors='coerce').fillna(0).astype('Int64')
    return prepared


def build_report_table(frame, columns):
    available = [column for column in columns if column in frame.columns]
    if not available:
        return pd.DataFrame()
    return frame[available].sort_values(available[: min(3, len(available))]).reset_index(drop=True)


def aggregate_by_code(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()

    def join_unique(series):
        values = [clean_text(value) for value in series if clean_text(value)]
        ordered = []
        for value in values:
            if value not in ordered:
                ordered.append(value)
        return ' | '.join(ordered) if ordered else None

    grouped = frame.groupby('Código', dropna=False).agg({
        'Nome': join_unique,
        'Professor': join_unique,
        'Horário': join_unique,
        'Curso': join_unique,
        'Fonte': join_unique,
    }).reset_index()
    grouped['nome_norm'] = grouped['Nome'].map(slug_text)
    grouped['professor_norm'] = grouped['Professor'].map(normalize_professor)
    grouped['horario_norm'] = grouped['Horário'].map(normalize_schedule)
    grouped['curso_tokens'] = grouped['Curso'].map(course_tokens)
    return grouped


def professor_missing(value):
    return normalize_professor(value) is None


def markdown_table(frame):
    if frame.empty:
        return 'Nenhuma inconsistência encontrada.'
    safe = frame.astype(object).where(pd.notna(frame), '')
    return safe.to_markdown(index=False)


def show_df(title, frame):
    display(Markdown(f'### {title}'))
    if frame.empty:
        display(Markdown('Nenhuma inconsistência encontrada.'))
    else:
        display(frame)


def section(title, frame):
    return f"{title}\n\n{markdown_table(frame)}\n"

In [3]:
DATA_ROOT = resolve_dados_path('Dados')
PLANILHA_PATH = find_numbers_file(DATA_ROOT)
REPORT_PATH = DATA_ROOT / 'comparacao.md'

planilha = prepare_dataset(load_numbers_spreadsheet(PLANILHA_PATH))

print('Diretório de dados resolvido:', DATA_ROOT)
print('Planilha localizada em:', PLANILHA_PATH)
print('Arquivo de saída:', REPORT_PATH)
print('Registros na planilha:', len(planilha))

display(planilha.head(10))

Diretório de dados resolvido: /Users/daltonreis/Library/Mobile Documents/com~apple~Numbers/Documents/DSC/2026_2/2026-06-01
Planilha localizada em: /Users/daltonreis/Library/Mobile Documents/com~apple~Numbers/Documents/DSC/2026_2/2026-06-01/TurmasDSC_2026_2.numbers
Arquivo de saída: /Users/daltonreis/Library/Mobile Documents/com~apple~Numbers/Documents/DSC/2026_2/2026-06-01/comparacao.md
Registros na planilha: 143


,Fonte,source_kind,Código,Nome,Curso,Professor,Espaço,Fase,Grupo,Horário,Créditos Teóricos,Créditos Práticos,Total de Créditos,concentrado_flag,nome_norm,professor_norm,horario_norm,curso_tokens,composite_key
0,TurmasDSC_2026_2.numbers,planilha,SIS.0124.00.001-3,Gestão da Informação,ADM-N,Angelica Karize Viecelli,_não_,7,A,NaN,2,0,2,False,GESTAO DA INFORMACAO,ANGELICA KARIZE VIECELLI,NaN,{ADM-N},SIS.0124.00.001-3|GESTAO DA INFORMACAO
1,TurmasDSC_2026_2.numbers,planilha,NaN,Alteridade e Direitos Humanos,BCC-N,_não_,_não_,4,A,Sex 12-13,2,0,2,False,ALTERIDADE E DIREITOS HUMANOS,NaN,SEX 12-13,{BCC-N},nan|ALTERIDADE E DIREITOS HUMANOS
2,TurmasDSC_2026_2.numbers,planilha,NaN,Alteridade e Direitos Humanos,BCC-N,_não_,_não_,4,B,Sex 14-15,2,0,2,False,ALTERIDADE E DIREITOS HUMANOS,NaN,SEX 14-15,{BCC-N},nan|ALTERIDADE E DIREITOS HUMANOS
3,TurmasDSC_2026_2.numbers,planilha,NaN,Alteridade e Direitos Humanos,BCC-N,_não_,_não_,4,C,Sex 12-13,2,0,2,False,ALTERIDADE E DIREITOS HUMANOS,NaN,SEX 12-13,{BCC-N},nan|ALTERIDADE E DIREITOS HUMANOS
4,TurmasDSC_2026_2.numbers,planilha,NaN,Ambiente Corporativo e Postura Profissional,BCC-N,_não_,_não_,3,A,Qui 14-15,2,0,2,False,AMBIENTE CORPORATIVO E POSTURA PROFISSIONAL,NaN,QUI 14-15,{BCC-N},nan|AMBIENTE CORPORATIVO E POSTURA PROFISSIONAL
5,TurmasDSC_2026_2.numbers,planilha,NaN,Ambiente Corporativo e Postura Profissional,BCC-N,_não_,_não_,3,B,Qui 12-13,2,0,2,False,AMBIENTE CORPORATIVO E POSTURA PROFISSIONAL,NaN,QUI 12-13,{BCC-N},nan|AMBIENTE CORPORATIVO E POSTURA PROFISSIONAL
6,TurmasDSC_2026_2.numbers,planilha,NaN,Diversidade e Sociedade,BCC-N,_não_,_não_,1,A,Sex 14-15,2,0,2,False,DIVERSIDADE E SOCIEDADE,NaN,SEX 14-15,{BCC-N},nan|DIVERSIDADE E SOCIEDADE
7,TurmasDSC_2026_2.numbers,planilha,NaN,História da Cultura Afro-brasileira e Indígena,BCC-N,_não_,_não_,4,A,Sex 14-15,2,0,2,False,HISTORIA DA CULTURA AFRO BRASILEIRA E INDIGENA,NaN,SEX 14-15,{BCC-N},nan|HISTORIA DA CULTURA AFRO BRASILEIRA E INDIGENA
8,TurmasDSC_2026_2.numbers,planilha,NaN,História da Cultura Afro-brasileira e Indígena,BCC-N,_não_,_não_,4,B,Sex 12-13,2,0,2,False,HISTORIA DA CULTURA AFRO BRASILEIRA E INDIGENA,NaN,SEX 12-13,{BCC-N},nan|HISTORIA DA CULTURA AFRO BRASILEIRA E INDIGENA
9,TurmasDSC_2026_2.numbers,planilha,NaN,História da Cultura Afro-brasileira e Indígena,BCC-N,_não_,_não_,4,C,Sex 14-15,2,0,2,False,HISTORIA DA CULTURA AFRO BRASILEIRA E INDIGENA,NaN,SEX 14-15,{BCC-N},nan|HISTORIA DA CULTURA AFRO BRASILEIRA E INDIGENA


In [4]:
pdfs_por_fonte = {
    filename: prepare_dataset(load_course_pdf(DATA_ROOT / filename, course_code))
    for filename, course_code in COURSE_FILE_MAP.items()
}
pdfs_por_fonte['DSC.pdf'] = prepare_dataset(load_dsc_pdf(DATA_ROOT / 'DSC.pdf'))
pdfs = pd.concat(pdfs_por_fonte.values(), ignore_index=True)

resumo_pdfs = pd.DataFrame(
    [(fonte, len(frame)) for fonte, frame in pdfs_por_fonte.items()],
    columns=['Fonte', 'Quantidade de turmas']
)

show_df('Resumo dos PDFs carregados', resumo_pdfs)
show_df('Amostra das turmas extraídas dos PDFs', pdfs.head(20))

### Resumo dos PDFs carregados

,Fonte,Quantidade de turmas
0,BCC_mat.pdf,20
1,BCC_not.pdf,86
2,BCD_not.pdf,15
3,SIS_not.pdf,23
4,DSC.pdf,100


### Amostra das turmas extraídas dos PDFs

,Fonte,source_kind,Código,Nome,Curso,Professor,Horário,Espaço,Fase,Grupo,Créditos Teóricos,Créditos Práticos,Total de Créditos,concentrado_flag,nome_norm,professor_norm,horario_norm,curso_tokens,composite_key
0,BCC_mat.pdf,pdf_curso,CMP.0167.02.002-7,Arquitetura de Computadores II,BCC-M,Miguel Alexandre Wisintainer,Seg 1/4,None,2,None,4,0,4,False,ARQUITETURA DE COMPUTADORES II,MIGUEL ALEXANDRE WISINTAINER,SEG 1/4,{BCC-M},CMP.0167.02.002-7|ARQUITETURA DE COMPUTADORES II
1,BCC_mat.pdf,pdf_curso,CMP.0168.00.002-0,Programação Orientada a Objetos,BCC-M,Luciana Pereira de Araújo Kohler,Qua 1/4; Qui 1/2,None,2,None,7,0,7,False,PROGRAMACAO ORIENTADA A OBJETOS,LUCIANA PEREIRA DE ARAUJO KOHLER,QUA 1/4; QUI 1/2,{BCC-M},CMP.0168.00.002-0|PROGRAMACAO ORIENTADA A OBJETOS
2,BCC_mat.pdf,pdf_curso,CMP.0169.00.002-3,Lógica para Computação,BCC-M,NaN,Sex 1/4,None,2,None,4,0,4,False,LOGICA PARA COMPUTACAO,NaN,SEX 1/4,{BCC-M},CMP.0169.00.002-3|LOGICA PARA COMPUTACAO
3,BCC_mat.pdf,pdf_curso,PDE.0007.00.000-9,Educação Física - Prática Desportiva II ***,BCC-M,NaN,NaN,None,2,None,2,0,2,False,EDUCACAO FISICA PRATICA DESPORTIVA II,NaN,NaN,{BCC-M},PDE.0007.00.000-9|EDUCACAO FISICA PRATICA DESPORTIVA II
4,BCC_mat.pdf,pdf_curso,SIS.0102.00.002-9,Banco de Dados,BCC-M,Luciana Pereira de Araújo Kohler,Qui 3/4,None,2,None,6,0,6,False,BANCO DE DADOS,LUCIANA PEREIRA DE ARAUJO KOHLER,QUI 3/4,{BCC-M},SIS.0102.00.002-9|BANCO DE DADOS
5,BCC_mat.pdf,pdf_curso,CMP.0170.02.002-4,Programação Web II,BCC-M,Luciana Pereira de Araújo Kohler,Seg 1/4,None,4,None,5,0,5,False,PROGRAMACAO WEB II,LUCIANA PEREIRA DE ARAUJO KOHLER,SEG 1/4,{BCC-M},CMP.0170.02.002-4|PROGRAMACAO WEB II
6,BCC_mat.pdf,pdf_curso,CMP.0178.00.002-2,Teoria dos Grafos,BCC-M,Patricia Kayser Vargas Mangan,Qui 1/4,None,4,None,4,0,4,False,TEORIA DOS GRAFOS,PATRICIA KAYSER VARGAS MANGAN,QUI 1/4,{BCC-M},CMP.0178.00.002-2|TEORIA DOS GRAFOS
7,BCC_mat.pdf,pdf_curso,CMP.0183.00.002-7,Compiladores,BCC-M,Joyce Martins,Qua 1/4,None,4,None,4,0,4,False,COMPILADORES,JOYCE MARTINS,QUA 1/4,{BCC-M},CMP.0183.00.002-7|COMPILADORES
8,BCC_mat.pdf,pdf_curso,HIS.0116.00.003-2,História da Cultura Afro-brasileira e Indígena (EAD),BCC-M,Ricardo Duwe,Sex 1/2 C 1/2 C 1/2 C 1/2 C,None,4,None,2,0,2,False,HISTORIA DA CULTURA AFRO BRASILEIRA E INDIGENA EAD,RICARDO DUWE,SEX 1/2 C 1/2 C 1/2 C 1/2 C,{BCC-M},HIS.0116.00.003-2|HISTORIA DA CULTURA AFRO BRASILEIRA E INDIGENA EAD
9,BCC_mat.pdf,pdf_curso,MAT.0131.00.002-6,Estatística,BCC-M,Luciane Zickuhr Tomelin,Ter 1/4,None,4,None,4,0,4,False,ESTATISTICA,LUCIANE ZICKUHR TOMELIN,TER 1/4,{BCC-M},MAT.0131.00.002-6|ESTATISTICA


In [5]:
base_columns = ['Fonte', 'Código', 'Nome', 'Curso', 'Professor', 'Horário', 'Créditos Teóricos', 'Créditos Práticos', 'Total de Créditos']

planilha_creditos_impares = planilha[planilha['Total de Créditos'].map(lambda value: pd.notna(value) and int(value) % 2 == 1)].copy()
planilha_creditos_impares['Tipo de inconsistência'] = 'Total de créditos ímpar'
planilha_creditos_impares = build_report_table(planilha_creditos_impares, base_columns + ['Tipo de inconsistência'])

planilha_concentrado = planilha[planilha['concentrado_flag']].copy()
planilha_concentrado['Tipo de inconsistência'] = 'Turma em concentrado'
planilha_concentrado = build_report_table(planilha_concentrado, base_columns + ['Tipo de inconsistência'])

planilha_sem_professor = planilha[planilha['Professor'].map(professor_missing)].copy()
planilha_sem_professor['Tipo de inconsistência'] = 'Turma sem professor'
planilha_sem_professor = build_report_table(planilha_sem_professor, base_columns + ['Tipo de inconsistência'])

prof_conf = planilha[planilha['professor_norm'].notna() & planilha['horario_norm'].notna()].copy()
prof_keys = prof_conf.groupby(['professor_norm', 'horario_norm']).size().reset_index(name='qtde')
prof_keys = prof_keys[prof_keys['qtde'] > 1]
conflitos_professor = prof_conf.merge(prof_keys[['professor_norm', 'horario_norm']], on=['professor_norm', 'horario_norm'])
conflitos_professor['Tipo de conflito'] = 'Conflito de professor'
conflitos_professor = build_report_table(conflitos_professor, ['Professor', 'Horário', 'Código', 'Nome', 'Curso', 'Tipo de conflito'])

space_conf = planilha[planilha['Espaço'].map(clean_text).notna() & planilha['horario_norm'].notna()].copy()
space_conf['espaco_norm'] = space_conf['Espaço'].map(slug_text)
space_keys = space_conf.groupby(['espaco_norm', 'horario_norm']).size().reset_index(name='qtde')
space_keys = space_keys[space_keys['qtde'] > 1]
conflitos_espaco = space_conf.merge(space_keys[['espaco_norm', 'horario_norm']], on=['espaco_norm', 'horario_norm'])
conflitos_espaco['Tipo de conflito'] = 'Conflito de espaço físico'
conflitos_espaco = build_report_table(conflitos_espaco, ['Espaço', 'Horário', 'Código', 'Nome', 'Curso', 'Tipo de conflito'])

phase_conf = planilha[
    planilha['Curso'].notna() &
    planilha['Fase'].notna() &
    planilha['Grupo'].notna() &
    planilha['horario_norm'].notna()
].copy()
phase_keys = phase_conf.groupby(['Curso', 'Fase', 'Grupo', 'horario_norm']).size().reset_index(name='qtde')
phase_keys = phase_keys[phase_keys['qtde'] > 1]
conflitos_fase = phase_conf.merge(phase_keys[['Curso', 'Fase', 'Grupo', 'horario_norm']], on=['Curso', 'Fase', 'Grupo', 'horario_norm'])
conflitos_fase['Tipo de conflito'] = 'Conflito de semestre/fase'
conflitos_fase = build_report_table(conflitos_fase, ['Curso', 'Fase', 'Grupo', 'Horário', 'Código', 'Nome', 'Tipo de conflito'])

show_df('Planilha - C1) Total de créditos ímpar', planilha_creditos_impares)
show_df('Planilha - C2) Turmas em concentrado', planilha_concentrado)
show_df('Planilha - C3) Turmas sem professor', planilha_sem_professor)
show_df('Planilha - D) Conflitos de professor', conflitos_professor)
show_df('Planilha - E) Conflitos de espaço físico', conflitos_espaco)
show_df('Planilha - F) Conflitos de semestre/fase', conflitos_fase)

### Planilha - C1) Total de créditos ímpar

Nenhuma inconsistência encontrada.

### Planilha - C2) Turmas em concentrado

,Fonte,Código,Nome,Curso,Professor,Horário,Créditos Teóricos,Créditos Práticos,Total de Créditos,Tipo de inconsistência
0,TurmasDSC_2026_2.numbers,CMP.0172.02.001-3,Eletiva II,BCC-N,Leandro Werner Ribeiro,Seg Concentrado:,4,0,4,Turma em concentrado
1,TurmasDSC_2026_2.numbers,CMP.0172.04.001-4,Eletiva IV,BCC-N,Miguel Alexandre Wisintainer,Seg Concentrado:,4,0,4,Turma em concentrado
2,TurmasDSC_2026_2.numbers,CMP.0172.05.001-0,Eletiva V,BCC-N,Guilherme Legal de Oliveria,Seg Concentrado:,2,0,2,Turma em concentrado
3,TurmasDSC_2026_2.numbers,SIS.0105.00.001-1,Inovação Tecnológica,BCC-N,Simone Erbs da Costa,Seg Concentrado:,2,0,2,Turma em concentrado
4,TurmasDSC_2026_2.numbers,SIS.0105.00.003-8,Inovação Tecnológica,BCC-N,Simone Erbs da Costa,Seg Concentrado:,2,0,2,Turma em concentrado
5,TurmasDSC_2026_2.numbers,NaN,Legislação em Informática,SIS-N,_não_,Seg Concentrado:,2,0,2,Turma em concentrado


### Planilha - C3) Turmas sem professor

...

,Professor,Horário,Código,Nome,Curso,Tipo de conflito
0,Alexander Roberto Valdameri,Seg 12-15,SIS.0125.00.001-7,Banco de Dados I,BCD-N,Conflito de professor
1,Alexander Roberto Valdameri,Seg 12-15,SIS.0125.00.002-5,Banco de Dados I,BCD-N,Conflito de professor
2,Everaldo Artur Grahl,Seg 12-15,SIS.0103.00.001-4,Projeto de Software,BCC-N,Conflito de professor
3,Everaldo Artur Grahl,Seg 12-15,SIS.0103.00.003-0,Projeto de Software,BCC-N,Conflito de professor
4,Simone Erbs da Costa,Seg Concentrado:,SIS.0105.00.001-1,Inovação Tecnológica,BCC-N,Conflito de professor
5,Simone Erbs da Costa,Seg Concentrado:,SIS.0105.00.003-8,Inovação Tecnológica,BCC-N,Conflito de professor


### Planilha - E) Conflitos de espaço físico

,Espaço,Horário,Código,Nome,Curso,Tipo de conflito
0,S-401,Qua 12-15,CDD.0002.00.002-8,Análise Exploratória de Dados,BCD-N,Conflito de espaço físico
1,S-401,Qua 12-15,MAT.0253.00.001-2,Fundamentos de Matemática,BCD-N,Conflito de espaço físico
2,S-401,Seg 12-15,SIS.0125.00.001-7,Banco de Dados I,BCD-N,Conflito de espaço físico
3,S-401,Seg 12-15,SIS.0125.00.002-5,Banco de Dados I,BCD-N,Conflito de espaço físico
4,S-401,Ter 12-15,CDD.0002.00.001-0,Análise Exploratória de Dados,BCD-N,Conflito de espaço físico
5,S-401,Ter 12-15,MAT.0218.00.001-0,Fundamentos Matemáticos,BCC-N,Conflito de espaço físico
6,_não_,Qua 12-15,CMP.0092.00.002-4,Teoria da Computação,BCC-N,Conflito de espaço físico
7,_não_,Qua 12-15,CMP.0169.00.004-0,Lógica para Computação,BCC-N,Conflito de espaço físico
8,_não_,Qua 12-15,CMP.0178.00.001-4,Teoria dos Grafos,BCC-N,Conflito de espaço físico
9,_não_,Qua 12-15,CMP.0190.00.002-9,Redes de Computadores,SIS-N,Conflito de espaço físico


### Planilha - F) Conflitos de semestre/fase

,Curso,Fase,Grupo,Horário,Código,Nome,Tipo de conflito
0,BCC-N,7,A,Seg Concentrado:,CMP.0172.05.001-0,Eletiva V,Conflito de semestre/fase
1,BCC-N,7,A,Seg Concentrado:,CMP.0172.04.001-4,Eletiva IV,Conflito de semestre/fase


In [6]:
pdfs_creditos_impares = pdfs[pdfs['Total de Créditos'].map(lambda value: pd.notna(value) and int(value) % 2 == 1)].copy()
pdfs_creditos_impares['Tipo de inconsistência'] = 'Total de créditos ímpar'
pdfs_creditos_impares = build_report_table(pdfs_creditos_impares, base_columns + ['Tipo de inconsistência'])

pdfs_concentrado = pdfs[pdfs['concentrado_flag']].copy()
pdfs_concentrado['Tipo de inconsistência'] = 'Turma em concentrado'
pdfs_concentrado = build_report_table(pdfs_concentrado, base_columns + ['Tipo de inconsistência'])

pdfs_sem_professor = pdfs[pdfs['Professor'].map(professor_missing)].copy()
pdfs_sem_professor['Tipo de inconsistência'] = 'Turma sem professor'
pdfs_sem_professor = build_report_table(pdfs_sem_professor, base_columns + ['Tipo de inconsistência'])

show_df('PDFs - C1) Total de créditos ímpar', pdfs_creditos_impares)
show_df('PDFs - C2) Turmas em concentrado', pdfs_concentrado)
show_df('PDFs - C3) Turmas sem professor', pdfs_sem_professor)

### PDFs - C1) Total de créditos ímpar

,Fonte,Código,Nome,Curso,Professor,Horário,Créditos Teóricos,Créditos Práticos,Total de Créditos,Tipo de inconsistência
0,BCC_mat.pdf,CMP.0168.00.002-0,Programação Orientada a Objetos,BCC-M,Luciana Pereira de Araújo Kohler,Qua 1/4; Qui 1/2,7,0,7,Total de créditos ímpar
1,BCC_mat.pdf,CMP.0170.02.002-4,Programação Web II,BCC-M,Luciana Pereira de Araújo Kohler,Seg 1/4,5,0,5,Total de créditos ímpar
2,BCC_not.pdf,CMP.0166.00.001-4,Introdução à Programação,BCC-N,Ricardo Voigt,Seg 12/13; Qua 12/15,7,0,7,Total de créditos ímpar
3,BCC_not.pdf,CMP.0168.00.001-1,Programação Orientada a Objetos,BCC-N,André Felipe Bürger,Seg 12/15; Qui 12/13,7,0,7,Total de créditos ímpar
4,BCC_not.pdf,CMP.0168.00.004-6,Programação Orientada a Objetos,BCC-N,André Felipe Bürger,Ter 12/15; Qui 14/15,7,0,7,Total de créditos ímpar
5,BCC_not.pdf,CMP.0170.01.001-0,Programação Web I,BCC-N,Marcos Rogério Cardoso,Qua 12/15,5,0,5,Total de créditos ímpar
6,BCC_not.pdf,CMP.0170.01.002-9,Programação Web I,BCC-N,Luciana Pereira de Araújo Kohler,Ter 12/15,5,0,5,Total de créditos ímpar
7,BCC_not.pdf,CMP.0170.02.001-6,Programação Web II,BCC-N,NaN,Ter 12/15,5,0,5,Total de créditos ímpar
8,BCC_not.pdf,CMP.0170.02.003-2,Programação Web II,BCC-N,NaN,Qua 12/15,5,0,5,Total de créditos ímpar
9,BCC_not.pdf,CMP.0170.02.004-0,Programação Web II,BCC-N,NaN,Seg 12/15,5,0,5,Total de créditos ímpar


### PDFs - C2) Turmas em concentrado

,Fonte,Código,Nome,Curso,Professor,Horário,Créditos Teóricos,Créditos Práticos,Total de Créditos,Tipo de inconsistência
0,DSC.pdf,CMP.0172.02.001-3,Eletiva II 133781Leandro Werner Ribeiro 4 0 12/15C 12/15C 12/15C 12/15C 12/15C 0,BCC-N,133781,Seg 0; Ter 12/15C; Qua 12/15C; Qui 12/15C; Sex 12/15C; Sab 12/15C,<NA>,4,4,Turma em concentrado
1,DSC.pdf,SIS.0105.00.001-1,Inovação Tecnológica 38443Simone Erbs da Costa 2 0 12/15C 12/15C 12/15C 12/15C 12/15C 1/4C 0,BCC-N,38443,Seg 0; Ter 12/15C; Qua 12/15C; Qui 12/15C; Sex 12/15C; Sab 12/15C,<NA>,2,2,Turma em concentrado
2,DSC.pdf,SIS.0105.00.003-8,Inovação Tecnológica 38443Simone Erbs da Costa 2 0 12/15C 12/15C 12/15C 12/15C 12/15C 1/4C 0,BCC-N,38443,Seg 0; Ter 12/15C; Qua 12/15C; Qui 12/15C; Sex 12/15C; Sab 12/15C,<NA>,2,2,Turma em concentrado


### PDFs - C3) Turmas sem professor

,Fonte,Código,Nome,Curso,Professor,Horário,Créditos Teóricos,Créditos Práticos,Total de Créditos,Tipo de inconsistência
0,BCC_mat.pdf,CMP.0169.00.002-3,Lógica para Computação,BCC-M,NaN,Sex 1/4,4,0,4,Turma sem professor
1,BCC_mat.pdf,CMP.0173.00.002-4,Inteligência Artificial,BCC-M,NaN,Sex 1/4,4,0,4,Turma sem professor
2,BCC_mat.pdf,CMP.0184.00.002-0,Sistemas Distribuídos,BCC-M,NaN,Qui 1/4,4,0,4,Turma sem professor
3,BCC_mat.pdf,PDE.0007.00.000-9,Educação Física - Prática Desportiva II ***,BCC-M,NaN,NaN,2,0,2,Turma sem professor
4,BCC_not.pdf,CMP.0084.00.001-0,Introdução à Computação,BCC-N,NaN,Seg 14/15,2,0,2,Turma sem professor
5,BCC_not.pdf,CMP.0169.00.001-5,Lógica para Computação,BCC-N,NaN,Ter 12/15,4,0,4,Turma sem professor
6,BCC_not.pdf,CMP.0169.00.004-0,Lógica para Computação,BCC-N,NaN,Qua 12/15,4,0,4,Turma sem professor
7,BCC_not.pdf,CMP.0170.02.001-6,Programação Web II,BCC-N,NaN,Ter 12/15,5,0,5,Turma sem professor
8,BCC_not.pdf,CMP.0170.02.003-2,Programação Web II,BCC-N,NaN,Qua 12/15,5,0,5,Turma sem professor
9,BCC_not.pdf,CMP.0170.02.004-0,Programação Web II,BCC-N,NaN,Seg 12/15,5,0,5,Turma sem professor


In [7]:
ausentes_cursos_frames = []
for filename, course_code in COURSE_FILE_MAP.items():
    pdf_df = pdfs_por_fonte[filename]
    planilha_keys = set(
        planilha[planilha['curso_tokens'].map(lambda tokens, current=course_code: current in tokens)]['composite_key'].dropna()
    )
    missing = pdf_df[~pdf_df['composite_key'].isin(planilha_keys)].copy()
    missing['Fonte PDF'] = filename
    ausentes_cursos_frames.append(missing)

ausentes_cursos = build_report_table(
    pd.concat(ausentes_cursos_frames, ignore_index=True),
    ['Fonte PDF', 'Código', 'Nome', 'Curso', 'Professor', 'Horário']
)

planilha_keys_geral = set(planilha['composite_key'].dropna())
ausentes_dsc = pdfs_por_fonte['DSC.pdf'][~pdfs_por_fonte['DSC.pdf']['composite_key'].isin(planilha_keys_geral)].copy()
ausentes_dsc['Fonte PDF'] = 'DSC.pdf'
ausentes_dsc = build_report_table(ausentes_dsc, ['Fonte PDF', 'Código', 'Nome', 'Curso', 'Professor', 'Horário'])

planilha_por_codigo = aggregate_by_code(planilha[planilha['Código'].notna()])
comparacoes = []
for source_name in sorted(pdfs['Fonte'].dropna().unique()):
    pdf_source = aggregate_by_code(pdfs[(pdfs['Fonte'] == source_name) & pdfs['Código'].notna()])
    merged = planilha_por_codigo.merge(pdf_source, on='Código', suffixes=(' na planilha', ' no PDF'))
    for _, row in merged.iterrows():
        base = {
            'Código': row['Código'],
            'Nome na planilha': row['Nome na planilha'],
            'Nome no PDF': row['Nome no PDF'],
            'Professor na planilha': row['Professor na planilha'],
            'Professor no PDF': row['Professor no PDF'],
            'Horário na planilha': row['Horário na planilha'],
            'Horário no PDF': row['Horário no PDF'],
            'Curso na planilha': row['Curso na planilha'],
            'Curso no PDF': row['Curso no PDF'],
            'Fonte PDF': source_name,
        }
        if row['nome_norm na planilha'] != row['nome_norm no PDF']:
            comparacoes.append({**base, 'Tipo de inconsistência': 'Nome diferente'})
        if row['professor_norm na planilha'] != row['professor_norm no PDF']:
            comparacoes.append({**base, 'Tipo de inconsistência': 'Professor diferente'})
        if row['horario_norm na planilha'] != row['horario_norm no PDF']:
            comparacoes.append({**base, 'Tipo de inconsistência': 'Horário diferente'})
        plan_courses = row['curso_tokens na planilha'] or set()
        pdf_courses = row['curso_tokens no PDF'] or set()
        if plan_courses != pdf_courses and not (pdf_courses and pdf_courses.issubset(plan_courses)) and not (plan_courses and plan_courses.issubset(pdf_courses)):
            comparacoes.append({**base, 'Tipo de inconsistência': 'Curso diferente'})

comparacoes = pd.DataFrame(comparacoes)
comparison_columns = [
    'Código', 'Nome na planilha', 'Nome no PDF', 'Professor na planilha', 'Professor no PDF',
    'Horário na planilha', 'Horário no PDF', 'Curso na planilha', 'Curso no PDF', 'Fonte PDF', 'Tipo de inconsistência'
]

nomes_diferentes = build_report_table(comparacoes[comparacoes['Tipo de inconsistência'] == 'Nome diferente'], comparison_columns) if not comparacoes.empty else pd.DataFrame(columns=comparison_columns)
professores_diferentes = build_report_table(comparacoes[comparacoes['Tipo de inconsistência'] == 'Professor diferente'], comparison_columns) if not comparacoes.empty else pd.DataFrame(columns=comparison_columns)
horarios_diferentes = build_report_table(comparacoes[comparacoes['Tipo de inconsistência'] == 'Horário diferente'], comparison_columns) if not comparacoes.empty else pd.DataFrame(columns=comparison_columns)
cursos_diferentes = build_report_table(comparacoes[comparacoes['Tipo de inconsistência'] == 'Curso diferente'], comparison_columns) if not comparacoes.empty else pd.DataFrame(columns=comparison_columns)

show_df('A) Turmas dos cursos ausentes na planilha', ausentes_cursos)
show_df('B) Turmas do DSC ausentes na planilha', ausentes_dsc)
show_df('G1) Nome diferente', nomes_diferentes)
show_df('G2) Professor diferente', professores_diferentes)
show_df('G3) Horário diferente', horarios_diferentes)
show_df('G4) Curso diferente', cursos_diferentes)

### A) Turmas dos cursos ausentes na planilha

,Fonte PDF,Código,Nome,Curso,Professor,Horário
0,BCC_mat.pdf,CMP.0177.00.002-9,Optativa I,BCC-M,Marcos Antonio Mattedi,Ter 1/4
1,BCC_mat.pdf,HIS.0116.00.003-2,História da Cultura Afro-brasileira e Indígena (EAD),BCC-M,Ricardo Duwe,Sex 1/2 C 1/2 C 1/2 C 1/2 C
2,BCC_mat.pdf,LET.0185.00.005-9,Produção Textual Acadêmica (EAD),BCC-M,Víctor César da Silva Nunes,Sex 1/4 C 1/4 C 1/4 C 1/4 C
3,BCC_mat.pdf,PDE.0007.00.000-9,Educação Física - Prática Desportiva II ***,BCC-M,NaN,NaN
4,BCC_mat.pdf,SOC.0200.00.003-7,Alteridade e Direitos Humanos (EAD),BCC-M,Oklinger Mantovaneli Junior,Sex 3/4 C 3/4 C 3/4 C 3/4 C
5,BCC_not.pdf,ADM.0522.00.001-3,Ambiente Corporativo e Postura Profissional,BCC-N,Michael Samir Dalfovo,Qui 14/15
6,BCC_not.pdf,ADM.0522.00.003-0,Ambiente Corporativo e Postura Profissional,BCC-N,Michael Samir Dalfovo,Qui 12/13
7,BCC_not.pdf,CMP.0177.00.001-0,Optativa I,BCC-N,Guilherme Legal de Oliveira,Qua 12/15
8,BCC_not.pdf,CMP.0177.00.003-7,Optativa I,BCC-N,Marcos Antonio Mattedi,Ter 12/15
9,BCC_not.pdf,DIR.0510.00.001-8,Legislação em Informática,BCC-N,NaN,Qui 12/13


### B) Turmas do DSC ausentes na planilha

,Fonte PDF,Código,Nome,Curso,Professor,Horário
0,DSC.pdf,CDD.0001.00.001-6,Introdução à Ciência de Dados 35512Marcos Antonio Mattedi 3 1 12/14 0,BCD-N,35512,Seg 1; Ter 12/14 15/15
1,DSC.pdf,CDD.0002.00.001-0,Análise Exploratória de Dados 160835Jonathan Gil Müller 2 3 12/13 0,BCD-N,160835,Seg 3; Qua 12/13 14/15
2,DSC.pdf,CDD.0002.00.002-8,Análise Exploratória de Dados 160835Jonathan Gil Müller 2 3 12/13 0,BCD-N,160835,Seg 3; Qui 12/13 14/15
3,DSC.pdf,CMP.0084.00.001-0,Introdução à Computação 2 0 14/15 0,BCC-N,NaN,Seg 0; Ter 14/15
4,DSC.pdf,CMP.0092.00.001-6,Teoria da Computação 10339Danton Cavalcanti Franco Junior 4 0 12/15 0,BCC-N,10339,Seg 0; Ter 12/15
5,DSC.pdf,CMP.0092.00.002-4,Teoria da Computação 10339Danton Cavalcanti Franco Junior 4 0 12/15 0,BCC-N,10339,Seg 0; Qui 12/15
6,DSC.pdf,CMP.0166.00.001-4,Introdução à Programação 100132Ricardo Voigt 6 1 12/13 12/15 0,BCC-N,100132,Seg 1; Ter 12/13; Qui 12/15
7,DSC.pdf,CMP.0167.01.001-3,Arquitetura de Computadores I 7671 Miguel Alexandre Wisintainer 4 0 12/15 0,BCC-N,7671,Seg 0; Sex 12/15
8,DSC.pdf,CMP.0167.02.001-9,Arquitetura de Computadores II 7671 Miguel Alexandre Wisintainer 4 0 12/15 0,BCC-N,7671,Seg 0; Qui 12/15
9,DSC.pdf,CMP.0167.02.002-7,Arquitetura de Computadores II 7671 Miguel Alexandre Wisintainer 4 0 1/4 0,BCC-M,7671,Seg 0; Ter 1/4


### G1) Nome diferente

...

,Código,Nome na planilha,Nome no PDF,Professor na planilha,Professor no PDF,Horário na planilha,Horário no PDF,Curso na planilha,Curso no PDF,Fonte PDF,Tipo de inconsistência
0,CDD.0001.00.001-6,Introdução à Ciência de Dados,Introdução à Ciência de Dados 35512Marcos Antonio Mattedi 3 1 12/14 0,Marcos Antônio Mattedi,35512,Seg 12-15,Seg 1; Ter 12/14 15/15,BCD-N,BCD-N,DSC.pdf,Professor diferente
1,CDD.0002.00.001-0,Análise Exploratória de Dados,Análise Exploratória de Dados 160835Jonathan Gil Müller 2 3 12/13 0,Jonathan Gil Müller,160835,Ter 12-15,Seg 3; Qua 12/13 14/15,BCD-N,BCD-N,DSC.pdf,Professor diferente
2,CDD.0002.00.002-8,Análise Exploratória de Dados,Análise Exploratória de Dados 160835Jonathan Gil Müller 2 3 12/13 0,Jonathan Gil Müller,160835,Qua 12-15,Seg 3; Qui 12/13 14/15,BCD-N,BCD-N,DSC.pdf,Professor diferente
3,CMP.0084.00.001-0,Introdução à Computação,Introdução à Computação,_PSPS,NaN,Seg 14-15,Seg 14/15,BCC-N,BCC-N,BCC_not.pdf,Professor diferente
4,CMP.0084.00.001-0,Introdução à Computação,Introdução à Computação 2 0 14/15 0,_PSPS,NaN,Seg 14-15,Seg 0; Ter 14/15,BCC-N,BCC-N,DSC.pdf,Professor diferente
5,CMP.0092.00.001-6,Teoria da Computação,Teoria da Computação 10339Danton Cavalcanti Franco Junior 4 0 12/15 0,Danton Cavalcanti Franco Junior,10339,Seg 12-15,Seg 0; Ter 12/15,BCC-N,BCC-N,DSC.pdf,Professor diferente
6,CMP.0092.00.002-4,Teoria da Computação,Teoria da Computação 10339Danton Cavalcanti Franco Junior 4 0 12/15 0,Danton Cavalcanti Franco Junior,10339,Qua 12-15,Seg 0; Qui 12/15,BCC-N,BCC-N,DSC.pdf,Professor diferente
7,CMP.0166.00.001-4,Introdução à Programação,Introdução à Programação 100132Ricardo Voigt 6 1 12/13 12/15 0,Ricardo Voigt,100132,Seg 12-13; Qua 12-15,Seg 1; Ter 12/13; Qui 12/15,BCC-N,BCC-N,DSC.pdf,Professor diferente
8,CMP.0167.01.001-3,Arquitetura de Computadores I,Arquitetura de Computadores I 7671 Miguel Alexandre Wisintainer 4 0 12/15 0,Miguel Alexandre Wisintainer,7671,Qui 12-15,Seg 0; Sex 12/15,BCC-N,BCC-N,DSC.pdf,Professor diferente
9,CMP.0167.02.001-9,Arquitetura de Computadores II,Arquitetura de Computadores II 7671 Miguel Alexandre Wisintainer 4 0 12/15 0,Miguel Alexandre Wisintainer,7671,Qua 12-15,Seg 0; Qui 12/15,BCC-N,BCC-N,DSC.pdf,Professor diferente


### G3) Horário diferente

,Código,Nome na planilha,Nome no PDF,Professor na planilha,Professor no PDF,Horário na planilha,Horário no PDF,Curso na planilha,Curso no PDF,Fonte PDF,Tipo de inconsistência
0,CDD.0001.00.001-6,Introdução à Ciência de Dados,Introdução à Ciência de Dados,Marcos Antônio Mattedi,Marcos Antonio Mattedi,Seg 12-15,Seg 12/14 15/15,BCD-N,BCD-N,BCD_not.pdf,Horário diferente
1,CDD.0001.00.001-6,Introdução à Ciência de Dados,Introdução à Ciência de Dados 35512Marcos Antonio Mattedi 3 1 12/14 0,Marcos Antônio Mattedi,35512,Seg 12-15,Seg 1; Ter 12/14 15/15,BCD-N,BCD-N,DSC.pdf,Horário diferente
2,CDD.0002.00.001-0,Análise Exploratória de Dados,Análise Exploratória de Dados,Jonathan Gil Müller,Jonathan Gil Müller,Ter 12-15,Ter 12/13 14/15,BCD-N,BCD-N,BCD_not.pdf,Horário diferente
3,CDD.0002.00.001-0,Análise Exploratória de Dados,Análise Exploratória de Dados 160835Jonathan Gil Müller 2 3 12/13 0,Jonathan Gil Müller,160835,Ter 12-15,Seg 3; Qua 12/13 14/15,BCD-N,BCD-N,DSC.pdf,Horário diferente
4,CDD.0002.00.002-8,Análise Exploratória de Dados,Análise Exploratória de Dados,Jonathan Gil Müller,Jonathan Gil Müller,Qua 12-15,Qua 12/13 14/15,BCD-N,BCD-N,BCD_not.pdf,Horário diferente
...,...,...,...,...,...,...,...,...,...,...,...
207,SIS.0124.00.001-3,Gestão da Informação,Gestão da Informação 184529Angelica Karize Viecelli 2 0 14/15 0,Angelica Karize Viecelli,184529,NaN,Seg 0; Ter 14/15,ADM-N,ADM-N,DSC.pdf,Horário diferente
208,SIS.0125.00.001-7,Banco de Dados I,Banco de Dados I,Alexander Roberto Valdameri,Alexander Roberto Valdameri,Seg 12-15,Seg 12/13 14/15,BCD-N,BCD-N,BCD_not.pdf,Horário diferente
209,SIS.0125.00.001-7,Banco de Dados I,Banco de Dados I 298Alexander Roberto Valdameri 2 4 12/13 0,Alexander Roberto Valdameri,298,Seg 12-15,Seg 4; Ter 12/13 14/15,BCD-N,BCD-N,DSC.pdf,Horário diferente
210,SIS.0125.00.002-5,Banco de Dados I,Banco de Dados I,Alexander Roberto Valdameri,Alexander Roberto Valdameri,Seg 12-15,NaN,BCD-N,BCD-N,BCD_not.pdf,Horário diferente


### G4) Curso diferente

Nenhuma inconsistência encontrada.

In [8]:
verificacoes_internas = pd.concat([planilha, pdfs], ignore_index=True)

c1 = build_report_table(
    verificacoes_internas[verificacoes_internas['Total de Créditos'].map(lambda value: pd.notna(value) and int(value) % 2 == 1)].assign(**{'Tipo de inconsistência': 'Total de créditos ímpar'}),
    base_columns + ['Tipo de inconsistência']
)
c2 = build_report_table(
    verificacoes_internas[verificacoes_internas['concentrado_flag']].assign(**{'Tipo de inconsistência': 'Turma em concentrado'}),
    base_columns + ['Tipo de inconsistência']
)
c3 = build_report_table(
    verificacoes_internas[verificacoes_internas['Professor'].map(professor_missing)].assign(**{'Tipo de inconsistência': 'Turma sem professor'}),
    base_columns + ['Tipo de inconsistência']
)

resumo = pd.DataFrame([
    ('A) Turmas dos cursos ausentes na planilha', len(ausentes_cursos)),
    ('B) Turmas do DSC ausentes na planilha', len(ausentes_dsc)),
    ('C1) Total de créditos ímpar', len(c1)),
    ('C2) Turmas em concentrado', len(c2)),
    ('C3) Turmas sem professor', len(c3)),
    ('D) Conflitos de professor', len(conflitos_professor)),
    ('E) Conflitos de espaço físico', len(conflitos_espaco)),
    ('F) Conflitos de semestre/fase', len(conflitos_fase)),
    ('G1) Nome diferente', len(nomes_diferentes)),
    ('G2) Professor diferente', len(professores_diferentes)),
    ('G3) Horário diferente', len(horarios_diferentes)),
    ('G4) Curso diferente', len(cursos_diferentes)),
], columns=['Tipo', 'Quantidade'])

report_parts = [
    '# Relatório de Comparação de Dados\n',
    section('## Resumo', resumo),
    section('## A) Turmas dos cursos ausentes na planilha', ausentes_cursos),
    section('## B) Turmas do DSC ausentes na planilha', ausentes_dsc),
    '## C) Verificações internas em todas as fontes\n\n',
    section('### C1) Total de créditos ímpar', c1),
    section('### C2) Turmas em concentrado', c2),
    section('### C3) Turmas sem professor', c3),
    section('## D) Conflitos de professor', conflitos_professor),
    section('## E) Conflitos de espaço físico', conflitos_espaco),
    section('## F) Conflitos de semestre/fase', conflitos_fase),
    '## G) Diferenças entre planilha e PDFs\n\n',
    section('### G1) Nome diferente', nomes_diferentes),
    section('### G2) Professor diferente', professores_diferentes),
    section('### G3) Horário diferente', horarios_diferentes),
    section('### G4) Curso diferente', cursos_diferentes),
]

report_text = '\n'.join(report_parts)
REPORT_PATH.write_text(report_text, encoding='utf-8')

show_df('Resumo final', resumo)
display(Markdown(f'Relatório gerado em: `{REPORT_PATH}`'))
display(Markdown('\n'.join(report_text.splitlines()[:20])))

### Resumo final

,Tipo,Quantidade
0,A) Turmas dos cursos ausentes na planilha,34
1,B) Turmas do DSC ausentes na planilha,100
2,C1) Total de créditos ímpar,17
3,C2) Turmas em concentrado,9
4,C3) Turmas sem professor,79
5,D) Conflitos de professor,6
6,E) Conflitos de espaço físico,52
7,F) Conflitos de semestre/fase,2
8,G1) Nome diferente,103
9,G2) Professor diferente,120


Relatório gerado em: `/Users/daltonreis/Library/Mobile Documents/com~apple~Numbers/Documents/DSC/2026_2/2026-06-01/comparacao.md`

# Relatório de Comparação de Dados

## Resumo

| Tipo                                      |   Quantidade |
|:------------------------------------------|-------------:|
| A) Turmas dos cursos ausentes na planilha |           34 |
| B) Turmas do DSC ausentes na planilha     |          100 |
| C1) Total de créditos ímpar               |           17 |
| C2) Turmas em concentrado                 |            9 |
| C3) Turmas sem professor                  |           79 |
| D) Conflitos de professor                 |            6 |
| E) Conflitos de espaço físico             |           52 |
| F) Conflitos de semestre/fase             |            2 |
| G1) Nome diferente                        |          103 |
| G2) Professor diferente                   |          120 |
| G3) Horário diferente                     |          212 |
| G4) Curso diferente                       |            0 |

## A) Turmas dos cursos ausentes na planilha